In [86]:
import json
from pathlib import Path
import numpy as np
import requests
import pandas as pd

In [87]:
league_id = 'm2cauar6l5yu7qdj'
season = '12'
latest_gw = 38

In [88]:
records = []

for period in range(1, latest_gw):  # periods 1–38
    url = f'https://www.fantrax.com/fxpa/req?leagueId={league_id}'

    headers = {
        "accept": "application/json",
        "content-type": "text/plain",
        "sec-ch-ua": "\"Not;A=Brand\";v=\"99\", \"Google Chrome\";v=\"139\", \"Chromium\";v=\"139\"",
        "sec-ch-ua-mobile": "?0",
        "sec-ch-ua-platform": "\"macOS\"",
        "referer": f"https://www.fantrax.com/fantasy/league/{league_id}/standings;view=SCHEDULE;timeframeType=BY_PERIOD;timeStartType=FROM_SEASON_START;period={period}"
    }
    
    payload = {
        "msgs": [
            {
                "method": "getStandings",
                "data": {
                    "leagueId": league_id,
                    "view": "REGULAR_SEASON",
                    "timeframeType": "BY_PERIOD",
                    "timeStartType": "FROM_SEASON_START",
                    "period": str(period)
                }
            }
        ],
        "uiv": 3,
        "refUrl": f"https://www.fantrax.com/fantasy/league/{league_id}/standings;view=SCHEDULE;timeframeType=BY_PERIOD;timeStartType=FROM_SEASON_START;period={period}",
        "dt": 1,
        "at": 0,
        "av": "0.0",
        "tz": "America/Los_Angeles",
        "v": "167.0.1"
    }

    r = requests.post(url, headers=headers, json=payload)
    r.raise_for_status()
    j = r.json()
    

    data = j["responses"][0]["data"]
    team_info = data.get("fantasyTeamInfo", {})

    # find the table for the current week table
    standings_tbl = next(
        (t for t in data.get("tableList", []) if t.get("caption") == f"Gameweek {period}"),
        None
    )
    if not standings_tbl:
        print(f"Period {period}: Head-to-head table not found")
        continue
    
    for row in standings_tbl['rows']:
        away_team   = row["cells"][0]["content"]
        away_score  = row["cells"][1]["content"]
        home_team   = row["cells"][2]["content"]
        home_score  = row["cells"][3]["content"]

        records.append({
            "gw": period,
            "away_team": away_team,
            "home_team": home_team,
            "away_score": float(away_score),
            "home_score": float(home_score),
        })
    
df = pd.DataFrame(records)

Period 37: Head-to-head table not found


In [89]:
# list of all teams
teams = sorted(df["home_team"].unique())

# Step 1 — Determine winner/loser
def get_result(row):
    if row["away_score"] > row["home_score"]:
        return row["away_team"], row["home_team"]
    elif row["home_score"] > row["away_score"]:
        return row["home_team"], row["away_team"]
    else:
        return None, None  # tie

df["winner"], df["loser"] = zip(*df.apply(get_result, axis=1))

# Step 2 — Build win records
wins = df.dropna(subset=["winner"]).loc[:, ["winner", "loser"]]
wins["count"] = 1

# Step 3 — Aggregate counts
win_counts = (
    wins.groupby(["winner", "loser"])["count"]
        .sum()
        .reset_index()
)

# Step 4 — Ensure all teams appear as losers for every winner
all_pairs = pd.MultiIndex.from_product([teams, teams], names=["winner", "loser"]).to_frame(index=False)
win_counts = all_pairs.merge(win_counts, on=["winner", "loser"], how="left").fillna(0).rename(
    columns={'count':'wins'}
)
win_counts["wins"] = win_counts["wins"].astype(int)

In [90]:
# Create mapping from team to index
team_index = {team: i for i, team in enumerate(teams)}

# Add x/y coordinates
win_counts["x"] = win_counts["loser"].map(team_index)
win_counts["y"] = win_counts["winner"].map(team_index)

In [91]:
win_counts.loc[win_counts["winner"] == win_counts["loser"], "wins"] = ""

/var/folders/fh/xklx_96541l_glf7cpxtt24m0000gq/T/ipykernel_52346/4098980889.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  win_counts.loc[win_counts["winner"] == win_counts["loser"], "wins"] = ""


In [92]:
win_counts.to_csv(f'data/output/h2h/season-{season}.csv', index=False)

In [93]:
chart_data = win_counts.to_csv(index=False)

In [94]:
name_to_short = {v["name"]: v["shortName"] for v in team_info.values()}

In [95]:
name_to_short

{'Mo Mané Mo Problems': 'davehop2',
 'I_De_Zerbit': 'DeZerbIt',
 'RATABE': 'RATABE',
 'Sa Lah Land': 'TJ',
 'Seanhampton FC': 'SFC',
 'Benford F.C.': 'BenMtz',
 'Bench Warmers FC': 'B-W FC.',
 'HappyHaalandays': 'HappyHaa'}

In [96]:
# hard code short names because these are too long
name_to_short = {
    'Mo Mané Mo Problems': 'MMMP',
    'I_De_Zerbit': 'DZ',
    'RATABE': 'RAT',
    'Sa Lah Land': 'TJ',
    'Seanhampton FC': 'SFC',
    'Benford F.C.': 'BenMtz',
    'Bench Warmers FC': 'BWFC',
    'HappyHaalandays': 'HH',
}

In [97]:
access_token = 'yXNpSzS8XN1dCvPgQUS2ofNz3IvncMH26TkRLZe2stW0fCDxXZAnCufjAOfRNiDc'
chart_id = 'uO64h'
base_url = 'https://api.datawrapper.de/v3/charts'
chart_url = f'{base_url}/{chart_id}'
chart_data_url = f'{chart_url}/data'
publish_url = f'{chart_url}/publish'

json_headers={
    'accept': '*/*',
    'Authorization':f'Bearer {access_token}',
    'content-type':'application/json' 
}
csv_headers={
    'accept': '*/*',
    'Authorization':f'Bearer {access_token}',
    'content-type':'text/csv'
}

Send latest data to Datawrapper

In [98]:
update_data_response = requests.put(chart_data_url, data=chart_data, headers=csv_headers)

Generate x-axis annotations

In [99]:
x_axis_annotions = []
for name, index in team_index.items():
    short_name = name_to_short[name]    
    anno_obj = {
        'dx': 0,
        'dy': 0,
        'id': f'x-{short_name}-{index}',
        'size': 10,
        'text': short_name,
        'align': 'bc',
        'width': 12.5,
        'position': {'x': index, 'y': '-0.5'},
    }
    x_axis_annotions.append(anno_obj)

Generate y-axis annotations

In [100]:
y_axis_annotions = []
for name, index in team_index.items():
    short_name = name_to_short[name]    
    anno_obj = {
        'dx': 0,
        'dy': 0,
        'id': f'y-{short_name}-{index}',
        'size': 10,
        'text': short_name,
        'align': 'mr',
        'width': 12.5,
        'position': {'x': '-0.55', 'y': index},
    }
    y_axis_annotions.append(anno_obj)

Set color scale responsive to max no. of wins

In [101]:
max_wins = max_wins = win_counts['wins'].where(win_counts['wins'] != '', 0).astype(int).max()
max_wins

np.int64(5)

In [102]:
WIN_COLORSETS = {
    0: ['#ffffcc'],
    1: ['#ffffcc', '#c2e699'],
    2: ['#ffffcc', '#c2e699', '#78c679'],
    3: ['#ffffcc', '#c2e699', '#78c679', '#238443'],
    4: ['#ffffcc', '#c2e699', '#78c679', '#238443', '#006837'],
    5: ['#ffffcc', '#d9f0a3', '#addd8e', '#78c679', '#31a354', '#006837'],
}

DEFAULT_COLORSET = ['#f7fcb9', '#addd8e', '#31a354']

def make_color_scheme(max_wins):
    colors = WIN_COLORSETS.get(max_wins, DEFAULT_COLORSET)

    return {
        "map": {str(i): colors[i] for i in range(len(colors))}
        | {"": "#ffffff"},
        "categoryLabels": {
            str(i): f"{i} wins" if i == max_wins and max_wins > 1 else str(i)
            for i in range(len(colors))
        }
    }

color_categories = make_color_scheme(max_wins)

Send to Datawrapper

In [104]:
payload = {
    "metadata": {
        "visualize": {
            "color-category": color_categories,
            "text-annotations": x_axis_annotions + y_axis_annotions
        }
    }
}

In [105]:
response = requests.patch(chart_url, json=payload, headers=json_headers)
response

<Response [200]>

In [106]:
publish_chart_response = requests.post(publish_url,headers=json_headers)
publish_chart_response

<Response [200]>

In [107]:
latest_chart_version = publish_chart_response.json()['data']['publicVersion']
latest_chart_version

8

In [108]:
filepath = Path('../_data/charts.json')
with filepath.open("r", encoding="utf-8") as f:
    charts = json.load(f)

charts[season]['h2h']['version'] = latest_chart_version

with filepath.open("w", encoding="utf-8") as f:
    json.dump(charts, f, indent=2)